In [1]:
import optuna

import csv
import logging
import sys
import time

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import typer

from torch.amp import GradScaler, autocast
from torch.optim import Adam
from torch.utils.data import DataLoader as TorchDataLoader, random_split
from torch_geometric.loader import DataLoader, DataLoader as PyGDataLoader
from tqdm import tqdm

from qqe.experiments.plotting import plot_training_curves
from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset, Regressor

from qqe.GNN.training.datasets import build_loaders, build_loaders_NN
from qqe.GNN.training.train_config import TrainConfig
from qqe.GNN.training.utils import (
    FamilyFeatureProjector,
    ProjectedDatasetWrapper,
    cache_root_paths,
    collect_files_path,
    evaluate_loss,
    unpack_supervised_batch,
)
from qqe.utils import configure_logger

logger = logging.getLogger(__name__)

In [2]:
def train_one_epoch(
    model: nn.Module,
    train_loader: PyGDataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    for batch in tqdm(train_loader, desc="Training", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = F.mse_loss(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_loader.dataset)

def evaluate(model: nn.Module, val_loader: PyGDataLoader, device: torch.device) -> float:
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating", leave=False):
            batch = batch.to(device)
            out = model(batch)
            loss = F.mse_loss(out, batch.y)
            total_loss += loss.item() * batch.num_graphs
    return total_loss / len(val_loader.dataset)

In [5]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    loss_type = trial.suggest_categorical("loss_type", ["mse", "huber"])
    gnn_heads = trial.suggest_categorical("gnn_heads", [4, 8, 16])
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-2, log=True)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128, 256])
    global_hidden = trial.suggest_categorical("global_hidden", [32, 64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    num_layers = trial.suggest_int("num_layers", 2, 6)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cfg = TrainConfig(
        epochs=40,
        lr=lr,
        loss_type=loss_type,
        training_mode="per_family",
        target="sre",
        show_progress=True,
        show_val_progress=False,
        log_batch_loss_every=5,
        heartbeat=60.0,
        epoch_warning=400.0,
    )

    family_filter = cfg.family if cfg.training_mode == "per_family" else None
    family_projection = cfg.family if cfg.training_mode == "per_family" else None
    data_paths = collect_files_path("../outputs/data", family=family_filter)

    train_loader, val_loader, test_loader, node_in_dim, global_in_dim, base_dataset = build_loaders(
        data_paths,
        seed = cfg.seed,
        train_split=cfg.train_split,
        val_split=cfg.val_split,
        global_feature_variant=cfg.global_feature_variant,
        node_feature_variant=cfg.node_feature_backend_variant,
        family_projection=family_projection,
    )
    model = GNN(
        node_in_dim=node_in_dim,
        global_in_dim=global_in_dim,
        gnn_hidden=hidden_dim,
        gnn_heads=gnn_heads,
        global_hidden= global_hidden,
        reg_hidden = hidden_dim,
        num_layers = num_layers,
        dropout_rate = dropout,
    )

    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_val = float("inf")

    for epoch in range(cfg.epochs):
        train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate(model, val_loader, device)

        trial.report(val_loss, step=epoch)

        if trial.should_prune():
            raise optuna.TrialPruned()
        
        best_val = min(best_val, val_loss)
    return best_val



In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="gnn",
    storage="sqlite:///gnn_optuna_study.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_startup_trials=10),
)
study.optimize(objective, n_trials=50)

[I 2026-04-23 15:15:39,972] Using an existing study with name 'gnn' instead of creating a new one.
[I 2026-04-24 01:13:46,558] Trial 1 finished with value: 0.07264843973517418 and parameters: {'lr': 0.0001329291894316216, 'loss_type': 'mse', 'gnn_heads': 4, 'weight_decay': 2.2310108018679158e-08, 'hidden_dim': 32, 'global_hidden': 32, 'dropout': 0.09170225492671691, 'num_layers': 3}. Best is trial 1 with value: 0.07264843973517418.
